# Gradient Boosting - FX Pairs

This notebook fits the published LightGBM menu to the same FX learning task used by the linear
models. The model's tree count is part of the configuration: every declared checkpoint is
persisted, predicted, checked for exact validation coverage, and exposed to backtesting. Rank
correlation describes predictions but does not select a checkpoint.

**Learning objectives**

- Resolve loss, tree capacity, and hardware settings before fitting.
- Persist every declared tree checkpoint from each fold's complete booster.
- Pass the full checkpoint population to the prediction catalog.

**Book reference**: Chapter 12, Section 12.2

**Prerequisites**: `02_labels`, `03_financial_features`, and `04_model_based_features`.

In [1]:
"""Fit and catalog the published gradient-boosting FX configurations."""

import polars as pl
import yaml

from case_studies.research import ExecutionTier, Study, plan_models
from case_studies.utils.gbm import gbm_checkpoint_iterations
from utils.modeling import load_configs
from utils.paths import get_case_study_dir

In [2]:
CASE_STUDY_ID = "fx_pairs"
PRIMARY_LABEL = ""
MAX_SYMBOLS = 0
MAX_FOLDS = 0
FORCE_RETRAIN = False
PREDICTION_SPLIT = "validation"
TRAIN_SAMPLE_FRAC = 1.0

## Select the task and execution tier

The setup file chooses GPU execution for canonical runs. Any fold, symbol, iteration, or sampling
reduction belongs to a preview identity and cannot enter an official population.

In [3]:
case_dir = get_case_study_dir(CASE_STUDY_ID)
setup = yaml.safe_load((case_dir / "config" / "setup.yaml").read_text())
labels = (
    [PRIMARY_LABEL]
    if PRIMARY_LABEL
    else [setup["labels"]["primary"], *setup["labels"].get("variants", [])]
)

if PREDICTION_SPLIT != "validation":
    raise ValueError("model selection uses validation predictions; holdout runs start from a lock")
if FORCE_RETRAIN:
    raise ValueError("valid boosters are replayed by identity; change the request to refit")
if not 0 < TRAIN_SAMPLE_FRAC <= 1:
    raise ValueError("TRAIN_SAMPLE_FRAC must be in (0, 1]")

reductions = {
    **({"folds": list(range(MAX_FOLDS))} if MAX_FOLDS else {}),
    **({"max_symbols": MAX_SYMBOLS} if MAX_SYMBOLS else {}),
    **({"train_sample_frac": TRAIN_SAMPLE_FRAC} if TRAIN_SAMPLE_FRAC < 1 else {}),
}
tier = ExecutionTier.PREVIEW if reductions else ExecutionTier.CANONICAL
study = Study.regenerate(CASE_STUDY_ID)

print(f"Labels: {', '.join(labels)}")
print(f"Execution tier: {tier.value}")
print(f"LightGBM device: {setup['modeling']['gbm']['device']}")

Labels: fwd_ret_1d, fwd_ret_5d, fwd_ret_21d
Execution tier: canonical
LightGBM device: cpu


## Build the declared request population

Huber presets declare a scale that the runner resolves against each training fold. This makes the
intended robust loss active at the return scale instead of reproducing squared error.

In [4]:
menu = [
    (label, config)
    for label in labels
    for config in load_configs(CASE_STUDY_ID, label, family="gbm")
]
requests = [
    study.model(
        family="gbm",
        label=label,
        config_name=config["config_name"],
        execution_tier=tier,
        preview_reductions=reductions,
        overrides={},
    )
    for label, config in menu
]

request_table = pl.DataFrame(
    {
        "label": [label for label, _ in menu],
        "config_name": [config["config_name"] for _, config in menu],
        "objective": [config["params"]["objective"] for _, config in menu],
        "max_iterations": [config["max_iterations"] for _, config in menu],
        "checkpoint_interval": [config["checkpoint_interval"] for _, config in menu],
        "declared_checkpoints": [len(gbm_checkpoint_iterations(config)) for _, config in menu],
    }
)
request_table

label,config_name,objective,max_iterations,checkpoint_interval,declared_checkpoints
str,str,str,i64,i64,i64
"""fwd_ret_1d""","""default_mse""","""regression""",500,50,10
"""fwd_ret_1d""","""default_mae""","""regression_l1""",500,50,10
"""fwd_ret_1d""","""default_huber""","""huber""",500,50,10
"""fwd_ret_1d""","""leaves_7_mse""","""regression""",500,50,10
"""fwd_ret_1d""","""leaves_7_mae""","""regression_l1""",500,50,10
…,…,…,…,…,…
"""fwd_ret_21d""","""leaves_31_mae""","""regression_l1""",500,50,10
"""fwd_ret_21d""","""leaves_31_huber""","""huber""",500,50,10
"""fwd_ret_21d""","""leaves_63_mse""","""regression""",500,50,10


## Declare every checkpoint before fitting

Each tree checkpoint is its own downstream configuration, so the population is the full cross
product of configurations and declared checkpoints. Planning resolves those identities without
fitting a booster, which is what lets a later failure show up as a missing member.

In [5]:
plan = plan_models(study, requests=requests)

configured = {(label, config["config_name"]) for label, config in menu}
planned = {(member.label, member.config_name) for member in plan.members}
if planned != configured:
    raise RuntimeError(
        "the plan does not match the configured GBM menu; "
        f"missing {sorted(configured - planned)}, unexpected {sorted(planned - configured)}"
    )

expected_checkpoints = sum(request_table.get_column("declared_checkpoints"))
if len(plan.expected_prediction_hashes) != expected_checkpoints:
    raise RuntimeError(
        f"the menu declares {expected_checkpoints} checkpoints, "
        f"the plan resolved {len(plan.expected_prediction_hashes)}"
    )

pl.DataFrame(
    {
        "label": [member.label for member in plan.members],
        "config_name": [member.config_name for member in plan.members],
        "checkpoint_kind": [member.checkpoint_kind for member in plan.members],
        "checkpoint_value": [member.checkpoint_value for member in plan.members],
        "prediction_hash": [member.prediction_hash for member in plan.members],
    }
)

label,config_name,checkpoint_kind,checkpoint_value,prediction_hash
str,str,str,i64,str
"""fwd_ret_1d""","""default_mse""","""iteration""",50,"""55afe0c17abb"""
"""fwd_ret_1d""","""default_mse""","""iteration""",100,"""868d44ca9c40"""
"""fwd_ret_1d""","""default_mse""","""iteration""",150,"""cd227080c99f"""
"""fwd_ret_1d""","""default_mse""","""iteration""",200,"""ddab626c248f"""
"""fwd_ret_1d""","""default_mse""","""iteration""",250,"""5a059a2d7bfe"""
…,…,…,…,…
"""fwd_ret_21d""","""leaves_63_huber""","""iteration""",300,"""c7a49586eda5"""
"""fwd_ret_21d""","""leaves_63_huber""","""iteration""",350,"""d230493dc350"""
"""fwd_ret_21d""","""leaves_63_huber""","""iteration""",400,"""1c81c20e1254"""


## Record the official population, then fit or replay every booster

A complete fold model contains all tree checkpoints. The shared runner can therefore reload a
valid booster and regenerate any declared checkpoint without retraining.

In [6]:
population = (
    plan.create_population(name=f"{CASE_STUDY_ID}:{'+'.join(labels)}:gbm")
    if tier is ExecutionTier.CANONICAL
    else None
)

execution = plan.run()
if len(execution.runs) != len(requests):
    raise RuntimeError("the GBM runner did not return every requested configuration")

catalog = execution.catalog_rows.sort("label", "config_name", "checkpoint_value")
if set(catalog.get_column("prediction_hash")) != set(plan.expected_prediction_hashes):
    raise RuntimeError("the published catalog differs from the population planned before fitting")
if catalog.filter(~pl.col("complete")).height:
    raise RuntimeError("partial checkpoint predictions cannot pass to backtesting")
if catalog.select("label", "config_name", "checkpoint_value").n_unique() != catalog.height:
    raise RuntimeError("each configuration and tree checkpoint must identify one prediction set")

catalog.select(
    "label",
    "config_name",
    "checkpoint_kind",
    "checkpoint_value",
    "complete",
    "ic_mean",
    "ic_t",
    "training_hash",
    "prediction_hash",
)

      fold 0: training n_train=25,780 n_val=5,140 trees=500 num_leaves=? obj=regression


      fold 0: done in 0s


      fold 0: training n_train=25,780 n_val=5,140 trees=500 num_leaves=? obj=regression_l1


      fold 0: done in 1s


      fold 0: training n_train=25,780 n_val=5,140 trees=500 num_leaves=? obj=huber


...

      fold 7: training n_train=17,860 n_val=5,160 trees=500 num_leaves=63 obj=regression_l1


      fold 7: done in 1s


      fold 7: training n_train=17,860 n_val=5,160 trees=500 num_leaves=63 obj=huber


      fold 7: done in 1s


label,config_name,checkpoint_kind,checkpoint_value,complete,ic_mean,ic_t,training_hash,prediction_hash
str,str,str,i64,bool,f64,f64,str,str
"""fwd_ret_1d""","""default_huber""","""iteration""",50,true,0.000258,0.026626,"""255f03d762dd""","""c2c20702ee0c"""
"""fwd_ret_1d""","""default_huber""","""iteration""",100,true,0.001042,0.135298,"""255f03d762dd""","""c00ee477cbd2"""
"""fwd_ret_1d""","""default_huber""","""iteration""",150,true,-0.000681,-0.077754,"""255f03d762dd""","""c78f58c08ff3"""
"""fwd_ret_1d""","""default_huber""","""iteration""",200,true,-0.0008,-0.09436,"""255f03d762dd""","""135c50121816"""
"""fwd_ret_1d""","""default_huber""","""iteration""",250,true,-0.00345,-0.386877,"""255f03d762dd""","""c8fc6cee76d6"""
…,…,…,…,…,…,…,…,…
"""fwd_ret_5d""","""leaves_7_mse""","""iteration""",300,true,-0.006738,-0.454625,"""c265b3f5dc59""","""6703c2da19f1"""
"""fwd_ret_5d""","""leaves_7_mse""","""iteration""",350,true,-0.005784,-0.401602,"""c265b3f5dc59""","""1953d1b089b9"""
"""fwd_ret_5d""","""leaves_7_mse""","""iteration""",400,true,-0.006922,-0.481139,"""c265b3f5dc59""","""d2f187d42ee1"""


## Verify fitted-state replay and catalog recovery

A second identical request must reuse the persisted boosters and recover the same checkpoint
identities. The catalog remains the only handoff the backtest notebook needs.

In [7]:
replayed = plan.run()
replayed_hashes = set(replayed.catalog_rows.get_column("prediction_hash"))
if replayed_hashes != set(catalog.get_column("prediction_hash")):
    raise RuntimeError("booster replay changed the declared checkpoint population")
if any(not diagnostic.get("cache_hit") for diagnostic in replayed.diagnostics):
    raise RuntimeError("an identical GBM request did not reuse its complete fitted state")

if population is not None:
    population.require_complete()
    print(f"Official prediction population: {population.hash}")
else:
    print("Preview checkpoints are isolated from official comparison and selection.")

Official prediction population: 183ef5c80011


## Key takeaways

- The requested loss and runtime settings are recorded before any model fits.
- Every declared tree checkpoint is a separate downstream configuration.
- Stored boosters reproduce checkpoint predictions without another training run.